# Community contribution

Generating a sports brochure - and then in Spanish! Many thanks for the contribution.

In [1]:
# imports

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
# A class to represent a Webpage

class Website:
    url: str
    title: str
    body: str
    links: List[str]

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
caps = Website("https://www.nhl.com/capitals/")
print(caps.get_contents())

Webpage Title:
Washington Capitals | Washington Capitals
Webpage Contents:
Skip to Main Content
Tickets
All Capitals Tickets
Single Game Tickets
2024-25 Season Tickets
Partial Plans
Promotions Schedule
Holiday Packs
GOVX Military & Govt Employee Discount Program
Special Ticket Offers
Group Tickets
VIP Seating
Account Manager
Tickets for Business
Using the New NHL Mobile App
NHL Ticket Exchange
Digital Ticketing
Capital One Arena
Transformation - The Vaults
Schedule
2024-25 Season Schedule
Practice Schedule
Where To Watch
Home Jersey Schedule
Schedule Sync & Download
Team
Capitals Roster
Capitals Prospects
Caps Alumni
Management
Coaching Staff
Equipment and Trainers
Staff Directory
Monumental Sports and Entertainment
AHL Hershey Bears
ECHL South Carolina Stingrays
News
Capitals News
Capitals Today
Dump N' Chase
Community News
Ted's Take
Video
All Video
Game Highlights
Mic'd Up
Capitals Locker Room
Caps365
Rinkside Update
Off the Ice
Coach's Corner
Caps Game Entertainment
Capitals Alumni

In [6]:
print(caps.links)

['#main-content', '/capitals', '/capitals', '/capitals/tickets/', 'https://www.ticketmaster.com/washington-capitals-tickets/artist/806039?brand=capitals&wt.mc_id=NHL_TEAM_WSH_SINGLE_GAME_TIX_LINK&utm_source=washcaps_tm&utm_medium=web_organic&utm_campaign=2425_sgb&utm_content=tickets_nav', 'https://www.nhl.com/capitals/club-red-365/', 'https://www.nhl.com/capitals/tickets/partial-plans', 'https://www.nhl.com/capitals/tickets/promotions', 'https://www.nhl.com/capitals/tickets/holiday-packs', 'https://www.govx.com/tickets/entertainers/30/washington-capitals/', 'https://www.nhl.com/capitals/tickets/offers', 'https://www.nhl.com/capitals/tickets/group-tickets', 'https://www.nhl.com/capitals/tickets/vip', 'https://am.ticketmaster.com/monumental/?wt.mc_id=NHL_TEAM_WSH_ACCOUNT_MANAGER_TIX&utm_source=washcaps_tm&utm_medium=web_organic&utm_campaign=2425sgb&utm_content=account_manager_tix', 'https://www.nhl.com/capitals/tickets/business', 'https://www.nhl.com/capitals/tickets/mobile-app-setup', '

In [7]:
# link_system_prompt = "You are provided with a list of links found on a webpage. \
# You are able to decide which of the links would be most relevant to include in a brochure about the team, \
# such as links to an About page, Team, News, Schedule, History, Stats pages.\n"
# link_system_prompt += "You should respond in JSON as in this example:"
# link_system_prompt += """
# {
#     "links": [
#         {"type": "about page", "url": "https://full.url/goes/here/about"},
#         {"type": "team page", "url": "https://full.url/goes/here/team"},
#         {"type": "news page": "url": "https://another.full.url/news"},
#         {"type": "schedule page": "url": "https://another.full.url/schedule"},
#         {"type": "history page": "url": "https://another.full.url/history"},
#         {"type": "stats page": "url": "https://another.full.url/stats"},
#         {"type": "standings page": "url": "https://another.full.url/standings"},
#     ]
# }
# """

In [ ]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the team, \
such as links to an About page, Team, News, Schedule, History, Stats pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "home page", "url": "https://www.ibp2.com/home"},
        {"type": "about page", "url": "https://www.ibp2.com/who-we-are"},
        {"type": "services page", "url": "https://www.ibp2.com/our-services"},
        {"type": "clients page": "url": "https://www.ibp2.com/clients"}
    ]
}
"""

In [8]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the team, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, Tickets, Video, Listen, Community, Fans, Youth Hockey, Shop, League, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [9]:
print(get_links_user_prompt(caps))

Here is the list of links on the website of https://www.nhl.com/capitals/ - please decide which of these are relevant web links for a brochure about the team, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, Tickets, Video, Listen, Community, Fans, Youth Hockey, Shop, League, email links.
Links (some might be relative links):
#main-content
/capitals
/capitals
/capitals/tickets/
https://www.ticketmaster.com/washington-capitals-tickets/artist/806039?brand=capitals&wt.mc_id=NHL_TEAM_WSH_SINGLE_GAME_TIX_LINK&utm_source=washcaps_tm&utm_medium=web_organic&utm_campaign=2425_sgb&utm_content=tickets_nav
https://www.nhl.com/capitals/club-red-365/
https://www.nhl.com/capitals/tickets/partial-plans
https://www.nhl.com/capitals/tickets/promotions
https://www.nhl.com/capitals/tickets/holiday-packs
https://www.govx.com/tickets/entertainers/30/washington-capitals/
https://www.nhl.com/capitals/tickets/offers
https://www.nhl.com/capitals/tickets/group-tickets
htt

In [10]:
def get_links(url):
    website = Website(url)
    completion = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = completion.choices[0].message.content
    return json.loads(result)

In [11]:
get_links("https://www.nhl.com/capitals/")

{'links': [{'type': 'about page', 'url': 'https://www.nhl.com/capitals'},
  {'type': 'team page', 'url': 'https://www.nhl.com/capitals/team'},
  {'type': 'news page', 'url': 'https://www.nhl.com/capitals/news'},
  {'type': 'schedule page', 'url': 'https://www.nhl.com/capitals/schedule'},
  {'type': 'history page', 'url': 'https://www.nhl.com/capitals/history'},
  {'type': 'stats page', 'url': 'https://www.nhl.com/capitals/stats'},
  {'type': 'standings page', 'url': 'https://www.nhl.com/capitals/standings'}]}

In [12]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [13]:
print(get_all_details("https://www.nhl.com/capitals/"))

KeyboardInterrupt: 

In [13]:
system_prompt = "You are a sports marketing analyst that analyzes the contents of several relevant pages from a sports team website \
and creates a short brochure about the team for prospective fans and players to recruit. Respond in markdown.\
Include details of team history, team culture, team news, and team stats if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the team in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:40_000] # Truncate if more than 40,000 characters
    return user_prompt

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("Washington Capitals", "https://www.nhl.com/capitals")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.nhl.com/capitals'}, {'type': 'team page', 'url': 'https://www.nhl.com/capitals/team/management'}, {'type': 'team page', 'url': 'https://www.nhl.com/capitals/team/coaching-staff'}, {'type': 'team page', 'url': 'https://www.nhl.com/capitals/team/equipment-training-staff'}, {'type': 'team page', 'url': 'https://www.nhl.com/capitals/team/staff'}, {'type': 'news page', 'url': 'https://www.nhl.com/capitals/news'}, {'type': 'schedule page', 'url': 'https://www.nhl.com/capitals/schedule'}, {'type': 'history page', 'url': 'https://www.nhl.com/capitals/history'}, {'type': 'stats page', 'url': 'https://www.nhl.com/capitals/stats'}, {'type': 'standings page', 'url': 'https://www.nhl.com/capitals/standings'}, {'type': 'roster page', 'url': 'https://www.nhl.com/capitals/roster'}, {'type': 'prospects page', 'url': 'https://www.nhl.com/capitals/prospects'}, {'type': 'alumni page', 'url': 'https://www.nhl.com/capitals/alumni'}]}


# Welcome to the Washington Capitals

## A Legacy in All Caps

The **Washington Capitals** are an esteemed professional ice hockey team that compete in the **NHL's Metropolitan Division**. Founded in **1974**, the Capitals have been a hallmark of passion, competitive spirit, and community engagement, marking over **50 years** in the league. They proudly celebrated their **50th Anniversary** during the 2023-2024 season, highlighting a rich history that includes memorable moments and storied legends.

### Team History

- **Founded**: 1974
- **First Playoff Appearance**: 1983
- **Stanley Cup Champions**: 2018
- **Presidents’ Trophy Winners**: 2010, 2016, 2017
- **Notable Players**: Alex Ovechkin, Nicklas Backstrom, and others have brought fame and success to the franchise.

Over the years, the Capitals have experienced their share of highs and lows, but their persistence culminated in a long-awaited **Stanley Cup win in 2018**, a historic moment that solidified their legacy in the NHL.

### Team Culture

The Capitals' culture is built on **community support, sportsmanship, and a commitment to winning**. The team is dedicated to fostering diversity and inclusion both on and off the ice. Initiatives include:

- **Caps in the Community**: Engaging with fans through various outreach programs.
- **Youth Hockey Development**: Programs aimed at cultivating the next generation of hockey players.
- **Military Commitment**: Special initiatives to honor and support military personnel and veterans.

The team’s mascot, **Slapshot**, and club initiatives like the **Caps Kids Club** further emphasize the Capitals’ commitment to creating a family-friendly environment for fans.

### Current News & Stats

As of **October 2023**, the Capitals have made significant strides in the new season, showcasing resilience and teamwork. Some key updates include:

- **Current Standing**: In the competitive race for the playoffs, the Capitals have maintained strong road performances, recently achieving **nine straight wins** on the road.
- **Recent Game Highlight**: The Capitals secured a **4-2 victory** over the Montreal Canadiens, proving their mettle with a come-from-behind victory.

#### Current Regular Season Stats (as of late 2023):

- **Overall Record**: 6 wins, 2 losses
- **Goals Scored**: 25
- **Goals Allowed**: 19

### Join Us!

The Capitals pride themselves on a **vibrant fan community** and a thrilling game-day experience at **Capital One Arena**. Fans can purchase tickets for the upcoming games, including single-game tickets, season tickets for 2024-25, and special promotions.

**Stay connected with us** through social media channels including [Facebook](https://www.facebook.com/washingtoncapitals), [Twitter](https://twitter.com/capitals), [Instagram](https://www.instagram.com/washingtoncapitals), and many others for the latest updates and exclusive content!

### Be part of the Capitals family and witness the pursuit of excellence on ice! 

🔗 [Visit our Official Website](https://www.washingtoncaps.com) for more information on tickets, merchandise, team news, and more! 

---
*Together, let's rally behind the Capitals and support the team as they continue to forge their path in the NHL! Go Caps!*

# Translate to Spanish

In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure in spanish of the team in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:40_000] # Truncate if more than 40,000 characters
    return user_prompt

In [18]:
create_brochure("Washington Capitals", "https://www.nhl.com/capitals")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.nhl.com/capitals'}, {'type': 'team page', 'url': 'https://www.nhl.com/capitals/team/management'}, {'type': 'news page', 'url': 'https://www.nhl.com/capitals/news'}, {'type': 'schedule page', 'url': 'https://www.nhl.com/capitals/schedule'}, {'type': 'history page', 'url': 'https://www.nhl.com/capitals/history'}, {'type': 'stats page', 'url': 'https://www.nhl.com/capitals/stats'}, {'type': 'standings page', 'url': 'https://www.nhl.com/capitals/standings'}, {'type': 'roster page', 'url': 'https://www.nhl.com/capitals/roster'}, {'type': 'prospects page', 'url': 'https://www.nhl.com/capitals/prospects'}, {'type': 'coaching staff page', 'url': 'https://www.nhl.com/capitals/team/coaching-staff'}]}


# Washington Capitals: ¡Únete a la Aventura!

## **Historia del Equipo**
Los **Washington Capitals** fueron fundados en 1974 y han tenido una rica historia en la National Hockey League (NHL). A lo largo de 50 años, el equipo ha crecido y se ha convertido en un rival formidable en la liga. Su momento culminante llegó en 2018, cuando finalmente alcanzaron la gloria al ganar la **Stanley Cup**, un logro que había eludido al equipo desde su inicio. Los Capitals han clasificado para los playoffs 32 veces, cimentando su lugar en la historia de la NHL.

## **Cultura del Equipo**
Los Capitals no son solo un equipo de hockey; son una comunidad. Apreciamos la diversidad, la inclusión y la elegancia en el juego. Esto se refleja en nuestras iniciativas comunitarias y programas que involucran a los aficionados. Un ambiente familiar se respira durante los juegos en el **Capital One Arena**, donde los aficionados pueden disfrutar de cada momento de la acción. ¡Esta es una oportunidad para ser parte de algo más grande que el hockey!

## **Últimas Noticias del Equipo**
- Ganamos **nueve partidos consecutivos de visitante** tras un emocionante regreso contra los Canadiens (último partido: 4-2).
- **Jakob Chychrun** ha sido nombrado el **Tercer Estrella de la Semana** en la NHL.
- Celebramos el **50 aniversario** de hockey de los Capitals con eventos especiales y recordatorios de nuestros momentos más destacados.
- ¡Únete a nuestro programa de capacitación y desarrollo de jugadores!

## **Estadísticas del Equipo**
- **Récord de la franquicia**: 1,616-1,155-201-185 (.573).
- La mejor actuación de todos los tiempos en una década con 465 victorias en la década del 2010.
- Con un porcentaje de puntos de .648, están entre los mejores equipos de la NHL en la historia en términos de efectividad.

## **Ven a vernos!**
Te invitamos a ser parte de nuestra próxima temporada (2024-25). Hay una variedad de opciones de **entradas** y **promociones** disponibles, ¡incluidas experiencias VIP! Ven y sé parte de la acción. Síguenos en nuestras redes sociales: [Facebook](https://www.facebook.com/washingtoncapitals), [Instagram](https://www.instagram.com/washingtoncapitals), y más.

**Washington Capitals: ¡A nuestro lado, el hockey nunca deja de ser emocionante!** 🏒✨